In [1]:
# --! include Koopa folder into PYTHONPATH --!

import os
import sys

thisdir  = os.getcwd()
koopadir = os.path.abspath(os.path.join(thisdir, '..', '..', 'external', 'Koopa'))
sys.path.append(koopadir)

# --! import python libraries and Koopa framework --!

import torch
import numpy as np
from exp.exp_main import Exp_Main
import random
import argparse
from   matplotlib import pyplot as plt


In [2]:
# --! build a parser for arguments --!

parser = argparse.ArgumentParser(description='Koopa for Time Series Forecasting')

# basic config
parser.add_argument('--is_training', type=int, required=True, default=1, help='status')
parser.add_argument('--model_id', type=str, required=True, default='test', help='model id')
parser.add_argument('--model', type=str, required=True, default='Koopa',
                    help='model name, options: [Koopa]')

# data loader
parser.add_argument('--data', type=str, required=True, default='ETTh2', help='dataset type')
parser.add_argument('--root_path', type=str, default='./data/ETT/', help='root path of the data file')
parser.add_argument('--data_path', type=str, default='ETTh2.csv', help='data file')
parser.add_argument('--features', type=str, default='M',
                    help='forecasting task, options:[M, S, MS]; M:multivariate predict multivariate, S:univariate predict univariate, MS:multivariate predict univariate')
parser.add_argument('--target', type=str, default='OT', help='target feature in S or MS task')
parser.add_argument('--freq', type=str, default='h',
                    help='freq for time features encoding, options:[s:secondly, t:minutely, h:hourly, d:daily, b:business days, w:weekly, m:monthly], you can also use more detailed freq like 15min or 3h')
parser.add_argument('--checkpoints', type=str, default='./checkpoints/', help='location of model checkpoints')

# forecasting task
parser.add_argument('--seq_len', type=int, default=96, help='input sequence length')
parser.add_argument('--label_len', type=int, default=48, help='start token length')
parser.add_argument('--pred_len', type=int, default=48, help='prediction sequence length')

# model define
parser.add_argument('--enc_in', type=int, default=7, help='encoder input size')
parser.add_argument('--dec_in', type=int, default=7, help='decoder input size')
parser.add_argument('--c_out', type=int, default=7, help='output size')
parser.add_argument('--dropout', type=float, default=0.05, help='dropout')
parser.add_argument('--embed', type=str, default='timeF',
                    help='time features encoding, options:[timeF, fixed, learned]')
parser.add_argument('--do_predict', action='store_true', help='whether to predict unseen future data')

# optimization
parser.add_argument('--num_workers', type=int, default=10, help='data loader num workers')
parser.add_argument('--itr', type=int, default=1, help='experiments times')
parser.add_argument('--train_epochs', type=int, default=10, help='train epochs')
parser.add_argument('--batch_size', type=int, default=32, help='batch size of train input data')
parser.add_argument('--patience', type=int, default=3, help='early stopping patience')
parser.add_argument('--learning_rate', type=float, default=0.001, help='optimizer learning rate')
parser.add_argument('--des', type=str, default='test', help='exp description')
parser.add_argument('--loss', type=str, default='mse', help='loss function')
parser.add_argument('--lradj', type=str, default='type1', help='adjust learning rate')
parser.add_argument('--use_amp', action='store_true', help='use automatic mixed precision training', default=False)

# GPU
parser.add_argument('--use_gpu', type=bool, default=True, help='use gpu')
parser.add_argument('--gpu', type=int, default=0, help='gpu')
parser.add_argument('--use_multi_gpu', action='store_true', help='use multiple gpus', default=False)
parser.add_argument('--devices', type=str, default='0,1,2,3', help='device ids of multile gpus')
parser.add_argument('--seed', type=int, default=2023, help='random seed')

# Koopa
parser.add_argument('--dynamic_dim', type=int, default=128, help='latent dimension of koopman embedding')
parser.add_argument('--hidden_dim', type=int, default=64, help='hidden dimension of en/decoder')
parser.add_argument('--hidden_layers', type=int, default=2, help='number of hidden layers of en/decoder')
parser.add_argument('--seg_len', type=int, default=48, help='segment length of time series')
parser.add_argument('--num_blocks', type=int, default=3, help='number of Koopa blocks')
parser.add_argument('--alpha', type=float, default=0.2, help='spectrum filter ratio')
parser.add_argument('--multistep', action='store_true', help='whether to use approximation for multistep K', default=False)

_StoreTrueAction(option_strings=['--multistep'], dest='multistep', nargs=0, const=True, default=False, type=None, choices=None, required=False, help='whether to use approximation for multistep K', metavar=None, deprecated=False)

In [3]:
# --! specify arguments --!

args = parser.parse_args(
    args=[
        "--is_training", "1",
        "--root_path", "../../data/baselines/",
        "--data_path", "ETTh2.csv",
        "--model_id", "ETTh2_96_48",
        "--model", "Koopa",
        "--data", "ETTh2",
        "--features", "S",
        "--seq_len", "96",
        "--pred_len", "48",
        "--seg_len", "48",
        "--dynamic_dim", "64",
        "--hidden_dim", "512",
        "--hidden_layers", "2",
        "--num_blocks", "4",
        "--enc_in", "1",
        "--dec_in", "1",
        "--c_out", "1",
        "--des", "Exp",
        "--learning_rate", "0.001",
        "--itr", "1",
        "--gpu", "0"
    ]
)
args.use_gpu = True if torch.cuda.is_available() and args.use_gpu else False
fix_seed = args.seed
random.seed(fix_seed)
torch.manual_seed(fix_seed)
np.random.seed(fix_seed)

if args.use_gpu:
    if args.use_multi_gpu:
        args.devices = args.devices.replace(' ', '')
        device_ids = args.devices.split(',')
        args.device_ids = [int(id_) for id_ in device_ids]
        args.gpu = args.device_ids[0]
    else:
        torch.cuda.set_device(args.gpu)

print('Args in experiment:')
print(args)

Args in experiment:
Namespace(is_training=1, model_id='ETTh2_96_48', model='Koopa', data='ETTh2', root_path='../../data/baselines/', data_path='ETTh2.csv', features='S', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=96, label_len=48, pred_len=48, enc_in=1, dec_in=1, c_out=1, dropout=0.05, embed='timeF', do_predict=False, num_workers=10, itr=1, train_epochs=10, batch_size=32, patience=3, learning_rate=0.001, des='Exp', loss='mse', lradj='type1', use_amp=False, use_gpu=False, gpu=0, use_multi_gpu=False, devices='0,1,2,3', seed=2023, dynamic_dim=64, hidden_dim=512, hidden_layers=2, seg_len=48, num_blocks=4, alpha=0.2, multistep=False)


In [4]:
Exp = Exp_Main

if args.is_training:
    for ii in range(args.itr):
        # setting record of experiments
        setting = '{}_{}_{}_ft{}_sl{}_pl{}_segl{}_dyna{}_h{}_l{}_nb{}_a{}_{}_{}'.format(
            args.model_id,
            args.model,
            args.data,
            args.features,
            args.seq_len,
            args.pred_len,
            args.seg_len,
            args.dynamic_dim,
            args.hidden_dim,
            args.hidden_layers,
            args.num_blocks,
            args.alpha,
            args.des, ii)

        exp = Exp(args)  # set experiments
        print('>>>>>>>start training : {}>>>>>>>>>>>>>>>>>>>>>>>>>>'.format(setting))
        exp.train(setting)

        print('>>>>>>>testing : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<'.format(setting))
        exp.test(setting)

        if args.do_predict:
            print('>>>>>>>predicting : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<'.format(setting))
            exp.predict(setting, True)

        torch.cuda.empty_cache()
else:
    ii = 0
    setting = '{}_{}_{}_ft{}_sl{}_pl{}_segl{}_dyna{}_h{}_l{}_nb{}_a{}_{}_{}'.format(
        args.model_id,
        args.model,
        args.data,
        args.features,
        args.seq_len,
        args.pred_len,
        args.seg_len,
        args.dynamic_dim,
        args.hidden_dim,
        args.hidden_layers,
        args.num_blocks,
        args.alpha,
        args.des, ii)

    exp = Exp(args)  # set experiments
    print('>>>>>>>testing : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<'.format(setting))
    exp.test(setting, test=1)
    torch.cuda.empty_cache()

Use CPU
train 8497
>>>>>>>start training : ETTh2_96_48_Koopa_ETTh2_ftS_sl96_pl48_segl48_dyna64_h512_l2_nb4_a0.2_Exp_0>>>>>>>>>>>>>>>>>>>>>>>>>>
train 8497
val 2833
test 2833
	iters: 100, epoch: 1 | loss: 0.1247241
	speed: 0.1045s/iter; left time: 266.6221s
	iters: 200, epoch: 1 | loss: 0.1311857
	speed: 0.0201s/iter; left time: 49.1667s
Epoch: 1 cost time: 63.46317911148071
Epoch: 1, Steps: 265 | Train Loss: 0.1455128 Vali Loss: 0.1571690 Test Loss: 0.1089695
Validation loss decreased (inf --> 0.157169).  Saving model ...
Updating learning rate to 0.001
	iters: 100, epoch: 2 | loss: 0.0931778
	speed: 1.8142s/iter; left time: 4147.2013s
	iters: 200, epoch: 2 | loss: 0.1070093
	speed: 0.0203s/iter; left time: 44.3010s
Epoch: 2 cost time: 63.57539200782776
Epoch: 2, Steps: 265 | Train Loss: 0.1329840 Vali Loss: 0.1382831 Test Loss: 0.1009472
Validation loss decreased (0.157169 --> 0.138283).  Saving model ...
Updating learning rate to 0.0005
	iters: 100, epoch: 3 | loss: 0.1001698
	speed: